In [ ]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”
# 🧩 Scenario Background

# You are working in a company called ABC Corp.

# Employees face issues like:

# VPN not working
# Printer not responding
# Software errors

# 👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

# 🤖 What this Bot Should Do

# When a user types a problem:

# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# 🧠 How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee




# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []  # This stores all tickets


# ============================================
# STEP 1: TOOL (MCP TOOL)
# ============================================

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP ORCHESTRATOR
# ============================================

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."


# ============================================
# STEP 5: RUN INTERACTIVE LOOP
# ============================================

print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 MCP Demo Started (Type 'exit' to stop)

Enter your query: Printer not responding

🧠 Agent received input: Printer not responding
➡️ Decision: No tool needed (AI response)
🤖 AI Response: Please describe your issue clearly.
Enter your query: VPN not working

🧠 Agent received input: VPN not working
➡️ Decision: Tool call required
📊 Extracted → Category: network, Priority: medium
📦 MCP Payload: {'issue': 'VPN not working', 'priority': 'medium', 'category': 'network'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: VPN not working
        Category: network
        Priority: medium
        


In [2]:
!pip install groq
import os
from groq import Groq
from google.colab import userdata

# Load API key securely
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS (REPLACES RULES)
# ============================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # fast + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "🤖 AI Response: No ticket required. Try basic troubleshooting."


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.2 MB/s eta 0:00:00
🚀 LLM MCP Helpdesk Started (type 'exit')

Enter issue: Printer is not responding

🧠 Agent received: Printer is not responding
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'Printer is not responding', 'priority': 'medium', 'category': 'general'}

✅ Ticket Created Successfully!

Ticket ID: INC1000
Issue: Printer is not responding
Category: general
Priority: medium

Enter issue: internet is slow

🧠 Agent received: internet is slow
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'internet is slow', 'priority': 'medium', 'category': 'general'}

✅ Ticket Created Successfully!

Ticket ID: INC1001
Issue: internet is slow
Category: general
Priority: medium



KeyboardInterrupt: Interrupted by user

In [7]:
# Install (only needed in Colab)
# !pip install groq

import os
from groq import Groq
import json

# ============================================
# 🔐 SET YOUR API KEY
# ============================================
# Option 1 (Colab secrets)
# from google.colab import userdata
# api_key = userdata.get("GROQ_API_KEY")

# Option 2 (local)
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

# ============================================
# 🗄️ DATABASE (In-Memory)
# ============================================

tickets_db = []

# ============================================
# 🛠️ TOOL: CREATE HR TICKET
# ============================================

def create_hr_ticket(issue, category, priority):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority,
        "status": "Open"
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# 📚 FAQ / INSTANT HELP (KNOWLEDGE BASE)
# ============================================

def get_instant_help(category):
    help_data = {
        "Payroll": "Try resetting your payroll password or access via company VPN.",
        "Policy": "You can check leave policy in the HR portal under 'Policies'.",
        "IT Setup": "Ensure VPN is connected and use company credentials.",
        "Training": "Training schedule is available on the onboarding dashboard.",
        "General": "Please contact HR for more details."
    }
    return help_data.get(category, "No help available.")


# ============================================
# 🧠 LLM ANALYSIS (MCP BRAIN)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an HR onboarding assistant.

Analyze the employee query and return ONLY JSON:

{{
  "create_ticket": true/false,
  "category": "Payroll/Policy/IT Setup/Training/General",
  "priority": "High/Medium/Low"
}}

Rules:
- Payroll login/salary issues → High
- IT access issues → Medium
- Policy/training questions → Low (no ticket)

User Query: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium"
        }

    return parsed


# ============================================
# 🤖 MCP AGENT
# ============================================

def hr_mcp_agent(user_input):

    print("\n🧠 Employee Query:", user_input)

    # Step 1: LLM decision
    decision = analyze_with_llm(user_input)
    print("🤖 LLM Decision:", decision)

    category = decision["category"]
    priority = decision["priority"]

    # Step 2: Instant help
    help_text = get_instant_help(category)

    # Step 3: Decision → Tool
    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "category": category,
            "priority": priority
        }

        print("📦 MCP Payload:", payload)

        ticket = create_hr_ticket(**payload)

        return f"""
✅ HR Ticket Created!

Ticket ID: {ticket['ticket_id']}
Category: {category}
Priority: {priority}

📌 Immediate Help:
{help_text}

➡️ HR team will contact you shortly.
"""

    else:
        return f"""
🤖 No Ticket Needed

📌 Instant Answer:
{help_text}
"""


# ============================================
# 🔁 RUN LOOP
# ============================================

print("🚀 Smart HR Onboarding Assistant Started (type 'exit')\n")

while True:
    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting HR Assistant...")
        break

    response = hr_mcp_agent(user_input)
    print(response)


🚀 Smart HR Onboarding Assistant Started (type 'exit')

Ask HR Bot: tI want to take leave

🧠 Employee Query: tI want to take leave
🤖 LLM Decision: {'create_ticket': True, 'category': 'General', 'priority': 'Medium'}
📦 MCP Payload: {'issue': 'tI want to take leave', 'category': 'General', 'priority': 'Medium'}

✅ HR Ticket Created!

Ticket ID: HR1000
Category: General
Priority: Medium

📌 Immediate Help:
Please contact HR for more details.

➡️ HR team will contact you shortly.



KeyboardInterrupt: Interrupted by user

In [6]:
# Install if using Colab
# !pip install groq

!pip install Groq
import os
from groq import Groq
import json

# ============================================
# 🔐 API KEY
# ============================================
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

# ============================================
# 🗄️ DATABASE
# ============================================
tickets_db = []

# ============================================
# 🛠️ TOOL: CREATE BANK TICKET
# ============================================

def create_bank_ticket(issue, category, priority):
    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority,
        "status": "Open"
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# 📚 INSTANT HELP (KNOWLEDGE BASE)
# ============================================

def get_bank_help(category):
    help_data = {
        "Card Services": "Check if your card is active, verify PIN, or try another terminal.",
        "Online Banking": "Try resetting your password or ensure correct login credentials.",
        "Loans": "You can check your loan status in the 'Loans' section of the app.",
        "Transactions": "Check transaction history or wait for 24 hours before raising dispute.",
        "General": "Please contact customer support for further assistance."
    }
    return help_data.get(category, "No help available.")


# ============================================
# 🧠 LLM ANALYSIS (MCP BRAIN)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are a banking support assistant.

Analyze the customer query and return ONLY JSON:

{{
  "create_ticket": true/false,
  "category": "Card Services/Online Banking/Loans/Transactions/General",
  "priority": "High/Medium"
}}

Rules:
- Card declined / fraud / transaction failure → High
- Login issues → Medium
- Loan queries → Medium (no urgent ticket unless critical)
- General questions → No ticket

User Query: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium"
        }

    return parsed


# ============================================
# 🤖 MCP AGENT
# ============================================

def banking_mcp_agent(user_input):

    print("\n🧠 Customer Query:", user_input)

    # Step 1: LLM Decision
    decision = analyze_with_llm(user_input)
    print("🤖 LLM Decision:", decision)

    category = decision["category"]
    priority = decision["priority"]

    # Step 2: Instant Help
    help_text = get_bank_help(category)

    # Step 3: Decision → Tool
    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "category": category,
            "priority": priority
        }

        print("📦 MCP Payload:", payload)

        ticket = create_bank_ticket(**payload)

        return f"""
✅ Support Ticket Created!

Ticket ID: {ticket['ticket_id']}
Category: {category}
Priority: {priority}

📌 Immediate Help:
{help_text}

➡️ Our banking team will contact you shortly.
"""

    else:
        return f"""
🤖 No Ticket Needed

📌 Instant Answer:
{help_text}
"""


# ============================================
# 🔁 RUN LOOP
# ============================================

print("🚀 Smart Banking Support Assistant Started (type 'exit')\n")

while True:
    user_input = input("Ask Bank Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting Banking Assistant...")
        break

    response = banking_mcp_agent(user_input)
    print(response)


🚀 Smart Banking Support Assistant Started (type 'exit')

Ask Bank Bot: Credit card not working

🧠 Customer Query: Credit card not working
🤖 LLM Decision: {'create_ticket': True, 'category': 'General', 'priority': 'Medium'}
📦 MCP Payload: {'issue': 'Credit card not working', 'category': 'General', 'priority': 'Medium'}

✅ Support Ticket Created!

Ticket ID: BNK1000
Category: General
Priority: Medium

📌 Immediate Help:
Please contact customer support for further assistance.

➡️ Our banking team will contact you shortly.



KeyboardInterrupt: Interrupted by user